# torch-Mesmer evaluation: original vs. Claude-edited weights

This notebook scores the segmentations of **torch-Mesmer** (`weights/model.pth`) and of its
weight-space-edited version (`model_CLAUDE.pth`) against the TissueNet v1.0 test-set ground truth.
It mirrors `evaluation_CP.ipynb`; the Cellpose datasets are replaced by Mesmer's two
**compartments**, `whole_cell` and `nuclear`, and the per-dataset breakdown by TissueNet's
**tissues**.

* **Inputs.** Predicted masks: `mesmer_results/first_results/{model,model_CLAUDE}/predicted_masks.npz`
  (keys `nuclear`, `whole_cell`; uint16, N×256×256). Images and ground truth:
  `mesmer_results/test_data/tissuenet_v1.0_test.npz` (`X`: ch 0 nuclear, ch 1 membrane; `y`:
  ch 0 whole-cell, ch 1 nuclear; `tissue_list`). The editing sandbox
  (`/mnt/volume/LLM_finetuning/torch-mesmer`) is only read, for the weights hashes and the
  integrity record. Tables go to `results/` and figures to `figures/` (prefix `mesmer_`).
* **Metrics.** The same per-image metric as `evaluation_CP.ipynb`: Cellpose's
  `average_precision` at IoU 0.50–0.95, AP = TP / (TP + FP + FN) and error rate =
  (FP + FN) / (TP + FN). The kernel has no `cellpose`, so the function is re-implemented here
  line for line (overlap matrix → IoU → Hungarian matching with Cellpose's cost). F1@0.5 =
  2TP / (2TP + FP + FN) is added to the summary because the Mesmer paper reports F1. Original
  and edited models are compared with a paired two-sided Wilcoxon signed-rank test.
* **Held-out images.** The Mesmer editing session drew its `dev` and `check` splits from the
  TissueNet *training* images (`train_NNNN`); it never saw the test set. Every test image is
  therefore held out, and the Inventory cell asserts this.

Kernel: **torch-mesmer** (`sinergia` env: numpy, scipy, pandas, matplotlib; no cv2 or cellpose).

In [ ]:
import hashlib
import json
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.ndimage import uniform_filter
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon

# The editing sandbox (torch-mesmer repo, weights, integrity record). READ-ONLY from here
# (CLAUDE.md rule 1): this notebook only reads from it.
SANDBOX = Path("/mnt/volume/LLM_finetuning/torch-mesmer")
WS = SANDBOX / "weight_space_work"

REPO = Path.cwd()
assert (REPO / "CLAUDE.md").is_file(), "start the notebook from the repo root"
MESMER = REPO / "mesmer_results"
PRED_ROOT = MESMER / "first_results"
TEST_NPZ = MESMER / "test_data" / "tissuenet_v1.0_test.npz"   # images + ground truth (gitignored)
RESULTS, FIGURES = REPO / "results", REPO / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

# figure style of the Cellpose-SAM paper (paper/cpsam/fig_utils.py), as in evaluation_CP.ipynb
mpl.rcParams.update({
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlelocation": "left", "axes.titleweight": "normal",
    "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42, "ps.fonttype": 42,
})
if any(f.name == "Arial" for f in font_manager.fontManager.ttflist):
    mpl.rcParams["font.family"] = "Arial"
pd.set_option("display.width", 200)

## Configuration

`MODELS` holds one entry per checkpoint: its prediction file, the weights it should have been made
with, and the sandbox copy of the predictions (used only to check that the copy here is
identical). `COMPARTMENTS` maps each prediction key to its ground-truth channel in `y`.

In [ ]:
COMPARTMENTS = {                  # prediction key -> (GT channel in y, label in figures)
    "whole_cell": (0, "whole-cell"),
    "nuclear":    (1, "nuclear"),
}
IOU_THRESHOLDS = np.round(np.arange(0.5, 1.0, 0.05), 2)   # as in paper/cpsam/benchmarks.py
IOU_MAIN = 0.5                                            # IoU of the violin plots

# Colours: one hue for Mesmer, a darker step of it for the Claude-edited weights (same scheme as the
# Cellpose notebook). Edited models are also hatched.
MODELS = {
    "mesmer": dict(
        label="Mesmer", family="mesmer", edited=False, color="#4a90d9",
        preds=PRED_ROOT / "model" / "predicted_masks.npz",
        weights=SANDBOX / "weights/model.pth",
        sandbox_copy=SANDBOX / "results/model/predicted_masks.npz"),
    "mesmer_CLAUDE": dict(
        label="Mesmer + Claude", family="mesmer", edited=True, color="#1f4f8f",
        preds=PRED_ROOT / "model_CLAUDE" / "predicted_masks.npz",
        weights=SANDBOX / "weights/model_CLAUDE.pth",
        sandbox_copy=SANDBOX / "results/model_CLAUDE/predicted_masks.npz"),
}
FAMILIES = {
    "mesmer": dict(dev_lists=sorted((WS / "splits").glob("*.txt")), integrity=WS / "integrity.json"),
}
HASH_WEIGHTS = True             # SHA-256 the weight files for the provenance check

## Inventory: ground truth, tissues, predictions

In [ ]:
with np.load(TEST_NPZ, allow_pickle=False) as z:
    X = z["X"]                                  # (N, 256, 256, 2) float32: ch 0 nuclear, ch 1 membrane
    Y = z["y"]                                  # (N, 256, 256, 2) int32:   ch 0 whole-cell, ch 1 nuclear
    TISSUE = z["tissue_list"].astype(str)
N = len(X)
assert Y.shape[:3] == X.shape[:3] and len(TISSUE) == N and Y.min() >= 0

PRED = {}
for m, cfg in MODELS.items():
    with np.load(cfg["preds"], allow_pickle=False) as z:
        PRED[m] = {c: z[c] for c in COMPARTMENTS}
    for c, a in PRED[m].items():
        assert a.shape == Y.shape[:3], f"{m}/{c}: {a.shape} vs GT {Y.shape[:3]}"

# The editing session's splits list only training-set images, so no test image was seen while editing.
seen = set()
for f in FAMILIES["mesmer"]["dev_lists"]:
    seen |= set(Path(f).read_text().split())
assert seen and all(s.startswith("train_") for s in seen), "a split names a non-training image"
print(f"editing splits {[Path(f).name for f in FAMILIES['mesmer']['dev_lists']]}: {len(seen)} images, "
      f"all from the training set -> all {N} test images are held out")

IMAGES = pd.DataFrame(dict(id=np.arange(N), tissue=TISSUE))
TISSUES = list(IMAGES.tissue.value_counts().index)        # largest first
for c, (ch, _) in COMPARTMENTS.items():
    IMAGES[f"gt_{c}"] = [len(np.unique(Y[i, ..., ch])) - 1 for i in range(N)]
    for m in MODELS:
        IMAGES[f"{m}_{c}"] = [len(np.unique(PRED[m][c][i])) - 1 for i in range(N)]

inv = IMAGES.groupby("tissue").agg(images=("id", "size"),
                                   **{col: (col, "sum") for col in IMAGES.columns if col[:3] in ("gt_", "mes")})
inv.loc["total"] = inv.sum()
print("\nobjects per tissue (ground truth and predictions):")
display(inv.loc[TISSUES + ["total"]])
for c in COMPARTMENTS:
    print(f"  {c:10s} images with no GT object: {(IMAGES[f'gt_{c}'] == 0).sum()}")

### Provenance check

Before scoring (CLAUDE.md rule 5), this cell checks for each model:

* the weights named in the prediction folder's `WEIGHTS.txt` are the configured weights, and both
  models used the same `image_mpp`;
* the predictions are newer than the weight file, so no prediction comes from an earlier edit
  state;
* the weight file's SHA-256 on disk matches the session's `integrity.json`;
* the prediction file here is byte-identical to the sandbox's own `results/` copy.

`None` means the check couldn't be made.

In [ ]:
def sha256(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


integ = json.loads(FAMILIES["mesmer"]["integrity"].read_text())
rows = []
for m, cfg in MODELS.items():
    lines = (cfg["preds"].parent / "WEIGHTS.txt").read_text().split("\n")
    rec_w = lines[0].strip()
    params = dict(l.split("=", 1) for l in lines[1:] if "=" in l)
    expected = integ.get("verified", {}).get("claude_sha256") if cfg["edited"] else integ.get("original_sha256")
    on_disk = sha256(cfg["weights"]) if HASH_WEIGHTS else None
    rows.append(dict(
        model=cfg["label"], preds=str(cfg["preds"].relative_to(REPO)),
        recorded_weights=Path(rec_w).name, image_mpp=params.get("image_mpp"),
        weights_match=Path(rec_w).resolve() == cfg["weights"].resolve(),
        newer_than_weights=min(cfg["preds"].stat().st_mtime, cfg["sandbox_copy"].stat().st_mtime
                               if cfg["sandbox_copy"].is_file() else np.inf) >= cfg["weights"].stat().st_mtime,
        sha_disk_eq_integrity=None if expected is None or on_disk is None else on_disk == expected,
        same_as_sandbox_copy=sha256(cfg["preds"]) == sha256(cfg["sandbox_copy"]) if cfg["sandbox_copy"].is_file() else None,
    ))
PROVENANCE = pd.DataFrame(rows)
display(PROVENANCE)
checks = ["weights_match", "newer_than_weights", "sha_disk_eq_integrity", "same_as_sandbox_copy"]
bad = PROVENANCE[checks].eq(False).any(axis=1)
if PROVENANCE.image_mpp.nunique() != 1:
    warnings.warn("PROVENANCE PROBLEM: the two models were run with different image_mpp")
if bad.any():
    warnings.warn(f"PROVENANCE PROBLEM in rows {list(PROVENANCE.index[bad])}: do not use these numbers before resolving it.")
else:
    print("provenance: no failed check")

## Score every prediction

Per image, compartment and model, AP/TP/FP/FN at all IoU thresholds. `average_precision` below is
Cellpose's `cellpose.metrics.average_precision` (`_label_overlap`, `_intersection_over_union`,
`_true_positive`), re-implemented without `fastremap`/`numba`. One deliberate difference: Cellpose
takes the number of GT objects as `masks_true.max()`, but 930 TissueNet whole-cell GT images have
gaps in their labels (2,483 missing ids), which Cellpose would count as missed cells. Here labels
are renumbered first, so FN counts real objects. On a 50-image check against `cellpose.metrics`,
TP and FP were identical and FN differed by exactly the label gaps (nuclear GT has none). The results go to
`results/mesmer_eval_per_image.csv`, one row per model × compartment × image × IoU threshold.

In [ ]:
def relabel(a):
    """Consecutive labels 0..n (0 stays background), like fastremap.renumber."""
    u, inv = np.unique(a, return_inverse=True)
    if u[0] != 0:
        inv = inv + 1
    return inv.reshape(a.shape).astype(np.int64), len(u) - (u[0] == 0)


def iou_matrix(gt, pred):
    """IoU between every GT label (rows) and predicted label (columns), background included."""
    ng, npred = gt.max() + 1, pred.max() + 1
    overlap = np.bincount((gt * npred + pred).ravel(), minlength=ng * npred).reshape(ng, npred)
    n_pred = overlap.sum(axis=0, keepdims=True)
    n_true = overlap.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        iou = overlap / (n_pred + n_true - overlap)
    iou[np.isnan(iou)] = 0.0
    return iou


def match_pairs(iou, th):
    """Matched (GT idx, pred idx), 0-based without background: cellpose.metrics._true_positive."""
    n_min = min(iou.shape)
    costs = -(iou >= th).astype(float) - iou / (2 * n_min)
    ti, pi = linear_sum_assignment(costs)
    ok = iou[ti, pi] >= th
    return ti[ok], pi[ok]


def average_precision(gt, pred, threshold):
    gt, n_true = relabel(gt)
    pred, n_pred = relabel(pred)
    tp = np.zeros(len(threshold))
    if n_pred > 0 and n_true > 0:
        iou = iou_matrix(gt, pred)[1:, 1:]
        tp = np.array([len(match_pairs(iou, th)[0]) for th in threshold], float)
    fp, fn = n_pred - tp, n_true - tp
    with np.errstate(divide="ignore", invalid="ignore"):
        ap = tp / (tp + fp + fn)
    return ap, tp, fp, fn


recs = []
for c, (ch, _) in COMPARTMENTS.items():
    for i in range(N):
        gt = Y[i, ..., ch]
        for m in MODELS:
            ap, tp, fp, fn = average_precision(gt, PRED[m][c][i], IOU_THRESHOLDS)
            for k, th in enumerate(IOU_THRESHOLDS):
                recs.append((m, c, i, TISSUE[i], th, ap[k], tp[k], fp[k], fn[k]))

EVAL = pd.DataFrame(recs, columns=["model", "compartment", "id", "tissue", "iou", "ap", "tp", "fp", "fn"])
EVAL[["tp", "fp", "fn"]] = EVAL[["tp", "fp", "fn"]].astype(int)
with np.errstate(divide="ignore", invalid="ignore"):
    EVAL["error_rate"] = (EVAL.fp + EVAL.fn) / (EVAL.tp + EVAL.fn)
    EVAL["f1"] = 2 * EVAL.tp / (2 * EVAL.tp + EVAL.fp + EVAL.fn)
EVAL["label"] = EVAL.model.map({m: c["label"] for m, c in MODELS.items()})
EVAL["edited"] = EVAL.model.map({m: c["edited"] for m, c in MODELS.items()})
EVAL["model"] = pd.Categorical(EVAL.model, categories=list(MODELS), ordered=True)
EVAL["compartment"] = pd.Categorical(EVAL.compartment, categories=list(COMPARTMENTS), ordered=True)
EVAL.to_csv(RESULTS / "mesmer_eval_per_image.csv", index=False)
print(f"{len(EVAL):,} rows -> {RESULTS / 'mesmer_eval_per_image.csv'}")
n_nan = EVAL[np.isclose(EVAL.iou, 0.5)].ap.isna().sum()
if n_nan:
    print(f"{n_nan} model x compartment x image rows have no GT and no predicted object (AP undefined, left out of means)")


def at_iou(df, th):
    return df[np.isclose(df.iou, th)]

## Summary tables

The first table gives means per image and total TP/FP/FN at IoU 0.5, per compartment, for all
test images and per tissue. The second table is the paired comparison of the original model with
its Claude-edited version on identical images. In it, Δ = edited − original. "improved", "worse"
and "same" count images by their change in AP@0.5, and *p* comes from a two-sided Wilcoxon
signed-rank test (images with zero difference are dropped, as in the Cellpose-SAM paper).

In [ ]:
def summarize(by):
    g = ["compartment"] + ([by] if by else []) + ["model"]
    e50 = at_iou(EVAL, 0.5)
    out = e50.groupby(g, observed=True).agg(images=("id", "size"), AP50=("ap", "mean"), F1_50=("f1", "mean"),
                                            error_rate50=("error_rate", "mean"),
                                            TP=("tp", "sum"), FP=("fp", "sum"), FN=("fn", "sum"))
    out["AP75"] = at_iou(EVAL, 0.75).groupby(g, observed=True).ap.mean()
    out["AP90"] = at_iou(EVAL, 0.9).groupby(g, observed=True).ap.mean()
    out["mAP50_95"] = EVAL.groupby(g, observed=True).ap.mean()
    return out[["images", "AP50", "AP75", "AP90", "mAP50_95", "F1_50", "error_rate50", "TP", "FP", "FN"]]


SUMMARY = summarize(None)
SUMMARY_TISSUE = summarize("tissue")
SUMMARY.round(4).to_csv(RESULTS / "mesmer_summary.csv")
SUMMARY_TISSUE.round(4).to_csv(RESULTS / "mesmer_summary_by_tissue.csv")
display(SUMMARY.round(4))
display(SUMMARY_TISSUE.round(4))


def paired_p(a, b):
    """Two-sided Wilcoxon signed-rank p (paper: scipy default, zero differences dropped)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    if not np.any(a[ok] != b[ok]):
        return np.nan                     # identical on every image
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return wilcoxon(a[ok], b[ok]).pvalue


def stars(p):
    return "n.s." if not np.isfinite(p) or p > 0.05 else "*" if p >= 0.01 else "**" if p >= 0.001 else "***"


def pairs():
    """(original, edited) model keys per family."""
    for fam in FAMILIES:
        for mo in [m for m, c in MODELS.items() if c["family"] == fam and not c["edited"]]:
            for me in [m for m, c in MODELS.items() if c["family"] == fam and c["edited"]]:
                yield fam, mo, me


def paired_row(e, mo, me, **keys):
    a = e[e.model == mo].set_index("id")
    b = e[e.model == me].set_index("id")
    ids = a.index.intersection(b.index)
    a, b = a.loc[ids], b.loc[ids]
    d = b.ap - a.ap
    return dict(**keys, images=len(ids),
                AP_original=a.ap.mean(), AP_claude=b.ap.mean(), dAP=d.mean(),
                improved=int((d > 0).sum()), worse=int((d < 0).sum()), same=int((d == 0).sum()),
                p_AP=paired_p(b.ap, a.ap),
                d_error_rate=(b.error_rate - a.error_rate).mean(), p_error=paired_p(b.error_rate, a.error_rate),
                dTP=int(b.tp.sum() - a.tp.sum()), dFP=int(b.fp.sum() - a.fp.sum()), dFN=int(b.fn.sum() - a.fn.sum()))


def paired_table(iou=IOU_MAIN):
    e = at_iou(EVAL, iou)
    rows = []
    for fam, mo, me in pairs():
        for c in COMPARTMENTS:
            ec = e[e.compartment == c]
            rows.append(paired_row(ec, mo, me, family=fam, compartment=c, tissue="all"))
            for t in TISSUES:
                rows.append(paired_row(ec[ec.tissue == t], mo, me, family=fam, compartment=c, tissue=t))
    return pd.DataFrame(rows)


PAIRED = paired_table()
PAIRED.to_csv(RESULTS / "mesmer_paired_tests.csv", index=False)
display(PAIRED.round({"AP_original": 4, "AP_claude": 4, "dAP": 4, "d_error_rate": 4}))

## Accuracy violin plots (Cellpose-SAM paper style)

Same layout as in `evaluation_CP.ipynb` (`supp_bench` in `paper/cpsam/figures.py`). The top row
shows the per-image error rate and the bottom row AP at IoU 0.5. Each group has one violin per
model and a thick line at the mean. Brackets compare the original model with its Claude-edited
version (paired Wilcoxon). Edited models are hatched.

The first figure groups by compartment (all test images); the next two split each compartment by
tissue.

In [ ]:
def style_violin(vp, color, edited):
    body = vp["bodies"][0]
    body.set_facecolor(mpl.colors.to_rgba(color, 0.35))
    body.set_edgecolor(color)
    body.set_linewidth(0.8)
    if edited:
        body.set_hatch("///")


def legend_handles(models):
    return [Patch(facecolor=mpl.colors.to_rgba(MODELS[m]["color"], 0.35), edgecolor=MODELS[m]["color"],
                  hatch="///" if MODELS[m]["edited"] else None, label=MODELS[m]["label"]) for m in models]


def violin_figure(groups, title, fname, iou=IOU_MAIN, save=True):
    """groups: list of (label, EVAL subset) -> one violin per model in each group."""
    groups = [(lab, at_iou(df, iou)) for lab, df in groups]
    n_v = len(groups) * len(MODELS)
    fig, axes = plt.subplots(2, 1, figsize=(1.6 + 0.75 * n_v + 0.6 * len(groups), 7.2), sharex=True,
                             gridspec_kw=dict(hspace=0.45))
    metrics_rows = [("error_rate", f"error rate @ {iou:g} IoU"), ("ap", f"average precision (AP) @ {iou:g} IoU")]
    xticks, xlabels = [], []
    lims = {}
    for row, (ax, (col, ylabel)) in enumerate(zip(axes, metrics_rows)):
        axin = ax.inset_axes([0, 1.0, 1, 0.16 if row else 0.32])
        k = 0.0
        for glabel, e in groups:
            start, pos = k, {}
            for m in MODELS:
                vals = e[e.model == m][col].to_numpy(float)
                vals = vals[np.isfinite(vals)]
                c = MODELS[m]["color"]
                if np.ptp(vals) > 0:
                    style_violin(ax.violinplot(vals, positions=[k], widths=0.6, showmeans=False,
                                               showextrema=False), c, MODELS[m]["edited"])
                ax.plot(k + 0.3 * np.array([-1, 1]), vals.mean() * np.ones(2), color=c, lw=3, solid_capstyle="butt")
                lims.setdefault(col, []).append((vals.min(), vals.max()))
                pos[m] = k
                if row == 0:
                    xticks.append(k)
                    xlabels.append(MODELS[m]["label"])
                k += 1
            # original vs edited, paired on identical images
            for fam, mo, me in pairs():
                a = e[e.model == mo].set_index("id")[col]
                b = e[e.model == me].set_index("id")[col]
                ids = a.index.intersection(b.index)
                p = paired_p(b.loc[ids], a.loc[ids])
                x0, x1 = pos[mo], pos[me]
                y = 0.15 if row else 0.08
                h = 0.25 if row else 0.12
                axin.plot([x0, x0, x1, x1], [y, y + h, y + h, y], lw=1, color="k")
                axin.text((x0 + x1) / 2, y + h + 0.02, stars(p), ha="center", va="bottom",
                          fontsize="small" if stars(p) == "n.s." else "large")
            if row == 0:
                n = len(e[e.model == list(MODELS)[0]])
                axin.text((start + k - 1) / 2, 0.62, f"{glabel}\nn={n:,}", ha="center", va="bottom",
                          fontsize="large")
            k += 1.5
        ax.set_ylabel(ylabel)
        ax.spines["bottom"].set_visible(False)
        ax.grid(True, color=0.8 * np.ones(3), lw=0.5, ls="--", axis="y")
        ax.set_axisbelow(True)
        ax.set_xlim(-0.6, k - 1.5 + 0.6)
        axin.set_xlim(ax.get_xlim())
        axin.set_ylim(0, 1)
        axin.axis("off")
    lo = min(v[0] for v in lims["ap"])
    hi = max(v[1] for v in lims["error_rate"])
    axes[1].set_ylim(min(0.4, np.floor(lo * 10) / 10), 1.0)
    axes[0].set_ylim(0, max(0.8, np.ceil(hi * 10) / 10))
    axes[1].set_xticks(xticks)
    axes[1].set_xticklabels(xlabels, rotation=45, ha="right")
    for ax in axes:
        ax.tick_params(axis="x", length=0)
    fig.suptitle(title, x=0.02, ha="left", y=1.04, fontstyle="italic", fontsize="large")
    axes[0].legend(handles=legend_handles(list(MODELS)), loc="upper left", bbox_to_anchor=(1.01, 1.0),
                   frameon=False, fontsize="small")
    if save:
        for ext in ("pdf", "png"):
            fig.savefig(FIGURES / f"{fname}.{ext}")
    plt.show()


violin_figure([(lab, EVAL[EVAL.compartment == c]) for c, (_, lab) in COMPARTMENTS.items()],
              "Performance on the TissueNet v1.0 test set (all images held out)", "mesmer_violins_compartments")
for c, (_, lab) in COMPARTMENTS.items():
    ec = EVAL[EVAL.compartment == c]
    violin_figure([(t, ec[ec.tissue == t]) for t in TISSUES],
                  f"TissueNet v1.0 test set, {lab} segmentation, by tissue", f"mesmer_violins_{c}_by_tissue")

## AP across IoU thresholds

The top row shows mean AP against IoU threshold, one column per compartment; the edited model is
dashed. The original and edited curves are close, so the bottom row plots their difference: mean
ΔAP (edited − original) on the same images, overall (thick) and per tissue (thin).

In [ ]:
TISSUE_COLORS = dict(zip(TISSUES, plt.get_cmap("tab10").colors))


def ap_curve_figure(save=True):
    comps = list(COMPARTMENTS)
    fig, axes = plt.subplots(2, len(comps), figsize=(3.6 * len(comps) + 2.2, 6.4), squeeze=False,
                             gridspec_kw=dict(height_ratios=[1.4, 1], hspace=0.4))
    for ax in axes[1, 1:]:
        ax.sharey(axes[1, 0])
    for j, c in enumerate(comps):
        e = EVAL[EVAL.compartment == c]
        ax = axes[0, j]
        for m in MODELS:
            g = e[e.model == m].groupby("iou").ap.mean()
            ax.plot(g.index, g.values, color=MODELS[m]["color"], lw=2, ls="--" if MODELS[m]["edited"] else "-",
                    label=MODELS[m]["label"])
        ax.set_title(COMPARTMENTS[c][1])
        ax.set_ylim(0, 1)
        ax.set_xlabel("IoU threshold")
        ax.grid(True, color=0.85 * np.ones(3), lw=0.5, ls="--")
        ax = axes[1, j]
        ax.axhline(0, color="0.4", lw=1)
        for fam, mo, me in pairs():
            a = e[e.model == mo].set_index(["id", "iou"]).ap
            b = e[e.model == me].set_index(["id", "iou"]).ap
            d = (b - a).dropna()
            tis = pd.Series(TISSUE, name="tissue")[d.index.get_level_values("id")].to_numpy()
            for t in TISSUES:
                dt = d[tis == t].groupby(level="iou").mean()
                ax.plot(dt.index, dt.values, color=TISSUE_COLORS[t], lw=1, alpha=0.8, label=t)
            dm = d.groupby(level="iou").mean()
            ax.plot(dm.index, dm.values, color=MODELS[me]["color"], lw=2.5, marker="o", ms=4,
                    label=f"all ({MODELS[me]['label']} − {MODELS[mo]['label']})")
        ax.set_xlabel("IoU threshold")
        ax.grid(True, color=0.85 * np.ones(3), lw=0.5, ls="--")
        if j == 0:
            axes[0, j].set_ylabel("mean AP")
            ax.set_ylabel("mean ΔAP (edited − original)")
    axes[0, -1].legend(loc="upper left", bbox_to_anchor=(1.03, 1), frameon=False, fontsize="small")
    axes[1, -1].legend(loc="upper left", bbox_to_anchor=(1.03, 1), frameon=False, fontsize="small")
    fig.suptitle("AP vs. IoU threshold (TissueNet v1.0 test, all images held out)", x=0.02, ha="left",
                 fontstyle="italic", fontsize="large")
    if save:
        for ext in ("pdf", "png"):
            fig.savefig(FIGURES / f"mesmer_ap_vs_iou.{ext}")
    plt.show()


ap_curve_figure()

## Segmentations side by side with the ground truth

Each row is one image, with these panels: the image (nuclear channel blue, membrane green), the
ground truth, and then one panel per model. In the model panels:

* **cyan**: predicted mask matched to a GT mask at IoU ≥ 0.5 (true positive)
* **red**: predicted mask with no match (false positive)
* **yellow**: GT mask that no prediction matched (false negative)

The matching is the same assignment `average_precision` uses, and the panels assert that their
counts agree with it. Outlines are drawn as boundary pixels (no cv2 in this kernel).

`show_segmentations(compartment, ids, zoom=..., focus=(original, edited))` can be called on any
image. With `zoom` set to a window size in pixels, the window centres on the area where the two
`focus` models disagree most (foreground or mask boundaries, so splits and merges count too).

In [ ]:
GT_COLOR = (0.8, 0.8, 0.3)    # paper's ground-truth outline colour
TP_COLOR = "#00e5ff"
FP_COLOR = "#ff3b30"


def load_image(i):
    """RGB for display: nuclear -> blue, membrane -> green, each scaled to its 1st-99.5th percentile."""
    out = np.zeros(X.shape[1:3] + (3,), np.float32)
    for rgb_c, ch in ((2, 0), (1, 1)):
        v = X[i, ..., ch]
        lo, hi = np.percentile(v, [1, 99.5])
        if hi > lo:
            out[..., rgb_c] = np.clip((v - lo) / (hi - lo), 0, 1)
    return out


def boundaries(labels):
    """Inner boundary pixels of every mask."""
    b = np.zeros(labels.shape, bool)
    b[:-1] |= labels[:-1] != labels[1:]
    b[1:] |= labels[1:] != labels[:-1]
    b[:, :-1] |= labels[:, :-1] != labels[:, 1:]
    b[:, 1:] |= labels[:, 1:] != labels[:, :-1]
    return b & (labels > 0)


def draw(ax, labels, color, keep=None):
    m = boundaries(labels)
    if keep is not None:
        m &= np.isin(labels, np.fromiter(keep, int, len(keep)))
    rgba = np.zeros(labels.shape + (4,), np.float32)
    rgba[m] = mpl.colors.to_rgba(color)
    ax.imshow(rgba, interpolation="nearest")


def match(gt, pred, th=0.5):
    """Matched (GT labels, predicted labels) in the original label values."""
    ug, up = np.unique(gt), np.unique(pred)
    ug, up = ug[ug > 0], up[up > 0]
    if not len(ug) or not len(up):
        return set(), set()
    g, _ = relabel(gt)
    p, _ = relabel(pred)
    ti, pi = match_pairs(iou_matrix(g, p)[1:, 1:], th)
    return set(ug[ti]), set(up[pi])


def window(shape, size, a=None, b=None):
    """(y0, y1, x0, x1): full image, or a size x size window where masks a and b differ most."""
    H, W = shape
    if size is None or size >= max(H, W):
        return 0, H, 0, W
    diff = None if a is None or b is None else ((a > 0) != (b > 0)) | (boundaries(a) != boundaries(b))
    if diff is not None and diff.any():   # foreground or mask-boundary differences (splits/merges)
        dens = uniform_filter(diff.astype(np.float32), size=size, mode="constant")
        cy, cx = np.unravel_index(np.argmax(dens), dens.shape)
    else:
        cy, cx = H // 2, W // 2
    y0 = int(np.clip(cy - size // 2, 0, max(H - size, 0)))
    x0 = int(np.clip(cx - size // 2, 0, max(W - size, 0)))
    return y0, min(y0 + size, H), x0, min(x0 + size, W)


def show_segmentations(compartment, ids, models=None, zoom=None, focus=None, iou=0.5, notes=None, save=None):
    """Rows = images; panels = image | ground truth | one per model. notes: {id: text} for row titles."""
    ch, clabel = COMPARTMENTS[compartment]
    models = models or list(MODELS)
    rows = []
    for i in ids:
        gt = Y[i, ..., ch]
        preds = {m: PRED[m][compartment][i] for m in models}
        fa, fb = (PRED[focus[0]][compartment][i], PRED[focus[1]][compartment][i]) if focus else (None, None)
        rows.append((i, gt, preds, window(gt.shape, zoom, fa, fb)))
    ncol = 2 + len(models)
    ratios = [(w[1] - w[0]) / (w[3] - w[2]) for *_, w in rows]
    fig, axes = plt.subplots(len(rows), ncol, figsize=(2.8 * ncol, sum(2.8 * r + 0.75 for r in ratios)),
                             squeeze=False, gridspec_kw=dict(height_ratios=ratios))
    for r, (i, gt, preds, (y0, y1, x0, x1)) in enumerate(rows):
        img = load_image(i)
        sl = (slice(y0, y1), slice(x0, x1))
        note = f"   {notes[i]}" if notes and i in notes else ""
        panels = [("image", None), (f"ground truth ({clabel})", None)] + [(MODELS[m]["label"], m) for m in models]
        for c, (title, m) in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(img[sl], interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
            for s_ in ax.spines.values():
                s_.set_visible(False)
            if c == 0:
                ax.set_title(f"test #{i} ({TISSUE[i]}){note}", fontsize="small")
                if zoom is not None:
                    ax.set_xlabel(f"x {x0}-{x1}, y {y0}-{y1}", fontsize="x-small")
                continue
            if m is None:
                draw(ax, gt[sl], GT_COLOR)
                ax.set_title(title, fontsize="small")
                continue
            mg, mp = match(gt, preds[m], iou)
            row = EVAL[(EVAL.model == m) & (EVAL.compartment == compartment) & (EVAL.id == i)
                       & np.isclose(EVAL.iou, iou)].iloc[0]
            assert len(mp) == row.tp, f"matching disagrees with average_precision for {m} {compartment} #{i}"
            all_p, all_g = set(np.unique(preds[m])[1:]), set(np.unique(gt)[1:])
            draw(ax, gt[sl], GT_COLOR, keep=all_g - mg)
            draw(ax, preds[m][sl], TP_COLOR, keep=mp)
            draw(ax, preds[m][sl], FP_COLOR, keep=all_p - mp)
            ax.set_title(title, fontsize="small")
            ax.set_xlabel(f"AP@{iou:g} {row.ap:.3f}   TP {row.tp}  FP {row.fp}  FN {row.fn}", fontsize="x-small")
    handles = [Line2D([], [], color=GT_COLOR, lw=1.5, label="ground truth / GT missed (FN)"),
               Line2D([], [], color=TP_COLOR, lw=1.5, label=f"prediction matched (IoU ≥ {iou:g})"),
               Line2D([], [], color=FP_COLOR, lw=1.5, label="prediction unmatched (FP)")]
    fig.tight_layout(rect=(0, 0.6 / fig.get_figheight(), 1, 1))
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize="small")
    if save:
        for ext in ("pdf", "png"):
            fig.savefig(FIGURES / f"{save}.{ext}")
    plt.show()


def delta_ap(compartment, original, edited, iou=0.5):
    e = at_iou(EVAL[EVAL.compartment == compartment], iou)
    a = e[e.model == original].set_index("id").ap
    b = e[e.model == edited].set_index("id").ap
    return (b - a).dropna().sort_values()

### Images the edit changed most

For each compartment, this shows the three test images whose AP@0.5 improved most and the two
that got worse most, if any. Set `EXAMPLES` to choose images by hand.

In [ ]:
EXAMPLES = {}            # e.g. {"nuclear": [12, 345]}; compartments left out are picked automatically
ZOOM = 128               # px window for the zoomed view (images are 256 x 256)

for fam, mo, me in pairs():
    for c in COMPARTMENTS:
        d = delta_ap(c, mo, me)
        print(f"{c}: {MODELS[me]['label']} vs {MODELS[mo]['label']}: "
              f"{(d > 0).sum()} improved, {(d < 0).sum()} worse, {(d == 0).sum()} unchanged (AP@0.5)")
        ids = EXAMPLES.get(c) or list(d[d > 0].sort_values(ascending=False).index[:3]) + list(d[d < 0].index[:2])
        display(d.loc[ids].rename("ΔAP@0.5").to_frame().T.round(4))
        notes = {i: f"ΔAP@0.5 {d[i]:+.3f}" for i in ids}
        show_segmentations(c, ids, notes=notes, save=f"mesmer_examples_{c}_full")
        show_segmentations(c, ids, zoom=ZOOM, focus=(mo, me), notes=notes, save=f"mesmer_examples_{c}_zoom")

### Any image

The same images as the sandbox's `run_mesmer_test.ipynb` overview (`comparison_examples.png`).

In [ ]:
show_segmentations("whole_cell", [0, 300, 700, 1100])
show_segmentations("nuclear", [0, 300, 700, 1100])

## Notes for interpreting the numbers

* **Not directly comparable with the Mesmer paper.** The paper reports DeepCell's object-level F1
  (with split/merge error categories) on TissueNet; here the metric is Cellpose's AP, with F1@0.5
  from the same matching in the summary table.
* **Inference settings.** Both checkpoints were run by the sandbox's `run_mesmer_test.ipynb` with
  identical code: `image_mpp=0.5` (the model's resolution; TissueNet is 0.5 µm/px),
  `compartment="both"`, repo-default post-processing. The provenance cell checks `image_mpp` from
  `WEIGHTS.txt`. These files don't record device or library versions (CLAUDE.md rule 5 asks for a
  manifest); this is a gap in the export, not something the notebook can fix.
* **Held-out.** The editing session's dev (24) and check (48) images come from the TissueNet
  *training* split, which the checkpoint has very likely seen during training. The test set
  scored here was never opened during editing, so there is no dev/check split to report within it.
* **Ground-truth location.** CLAUDE.md asks for ground truth under `ground_truth/`. The TissueNet
  test labels currently live in `mesmer_results/test_data/` (gitignored); change `TEST_NPZ` if they
  are moved.